## Phase 6: Missing Value Handling

In [1]:
# Import pandas for DataFrame missing value analysis and handling
import pandas as pd

# Load cleaned metadata CSV from previous data cleaning phase
cleaned_csv_path = "data/processed/utkface_cleaned.csv"
df = pd.read_csv(cleaned_csv_path)

print(f"Successfully loaded '{cleaned_csv_path}' ({len(df)} rows, {len(df.columns)} columns).")

Successfully loaded 'data/processed/utkface_cleaned.csv' (23705 rows, 5 columns).


In [2]:
# Calculate count and percentage of missing values per column
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

# Construct a summary DataFrame for Before Treatment inspection
before_treatment_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing Percentage (%)': missing_percentage
})

print("=== BEFORE TREATMENT: MISSING VALUE ANALYSIS ===")
print(before_treatment_df)
print(f"\nTotal missing values across all columns: {df.isnull().sum().sum()}")

=== BEFORE TREATMENT: MISSING VALUE ANALYSIS ===
            Missing Count  Missing Percentage (%)
image_name              0                     0.0
age                     0                     0.0
gender                  0                     0.0
race                    0                     0.0
filepath                0                     0.0

Total missing values across all columns: 0


### Missing Value Treatment Decision Logic

The decision on missing value treatment depends entirely on the empirical findings from the **Before Treatment** check:
- **Empirical Finding:** Exactly **0 missing values (0.0%)** exist across all columns (`image_name`, `age`, `gender`, `race`, `filepath`).
- **Technical Rationale:** The dataset was constructed by directly parsing image filenames (`[age]_[gender]_[race]_[date].jpg`). Any filename that failed to parse into valid integer attributes was already caught, logged, and isolated during Phase 4/5. Consequently, there is no mechanism by which `NaN` or null values could be introduced into this structured dataset.
- **Action Taken:** No `fillna()` or `dropna()` operations are required or applied. Modifying or fabricating synthetic missing values when none exist is avoided to maintain strict data integrity.

In [3]:
# Total missing values count across entire DataFrame
total_missing_values = df.isnull().sum().sum()

# Conditional treatment logic: Only apply fillna if missing values actually exist (> 0)
if total_missing_values > 0:
    print(f"Detected {total_missing_values} missing values. Applying appropriate treatment...")
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            if col == 'age':
                # Numerical feature: Impute with median to remain robust against skewed age distribution
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                print(f"Imputed column '{col}' with median: {median_val}")
            elif col in ['gender', 'race']:
                # Categorical feature: Impute with mode (most frequent discrete code)
                mode_val = df[col].mode()[0]
                df[col] = df[col].fillna(mode_val)
                print(f"Imputed column '{col}' with mode: {mode_val}")
else:
    print("No missing values detected. No treatment required.")

No missing values detected. No treatment required.


In [4]:
# Re-run missing value verification post-treatment
after_missing_count = df.isnull().sum()
after_missing_percentage = (df.isnull().sum() / len(df)) * 100

after_treatment_df = pd.DataFrame({
    'Missing Count': after_missing_count,
    'Missing Percentage (%)': after_missing_percentage
})

print("=== AFTER TREATMENT - VERIFICATION ===")
print(after_treatment_df)
print(f"\nVerification Status: {'Passed' if df.isnull().sum().sum() == 0 else 'Failed'} ({df.isnull().sum().sum()} missing values confirmed).")

=== AFTER TREATMENT - VERIFICATION ===
            Missing Count  Missing Percentage (%)
image_name              0                     0.0
age                     0                     0.0
gender                  0                     0.0
race                    0                     0.0
filepath                0                     0.0

Verification Status: Passed (0 missing values confirmed).


In [5]:
# Export DataFrame to data/processed/utkface_missing_handled.csv
output_path = "data/processed/utkface_missing_handled.csv"
df.to_csv(output_path, index=False)

print(f"DataFrame exported successfully to '{output_path}' ({len(df)} rows).")

DataFrame exported successfully to 'data/processed/utkface_missing_handled.csv' (23705 rows).
